# NB06 v3 — Build03 Current Law Checker · Cellar SPARQL Tool → Q8 Official Test

**Scientific AI Builder · Regulatory Agent · 2026-10-19**

## Objective
Verify whether **official-source current-law checking** improves Q8 relative to **frozen Build02 static RAG**, while retaining the exact NB05 v2 retrieval implementation and the NB04b common benchmark context.

- **Build02 stays frozen**: 5 local legislation files, `chunks_legal_aware.json`, NB03 BM25/Dense/RRF/diversification, benchmark Q1–Q10, NB04b answers.
- **Build03 intervention**: add a second function tool, `check_current_law(regulation)`, which fetches **only official EUR-Lex pages** at run time and produces evidence records. Existing `search_regulation()` is unchanged.
- **No implied real-time legal certainty**: detecting an updated consolidated version does *not* prove current applicability, article-level content changes or effective dates. When retrieval/metadata fails → `unverified`, not `unchanged`.
- **Q8 is the planned structural test**: “Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?” (fixed wording; no gold checklist in agent prompt).
- No unrestricted browsing, no web search engine dependency, no scraping of arbitrary sites, no fake test results.

**Run order:** dependencies → asset/config audit → exact NB03 retriever → **50/50 stop gate** → offline parser unit tests → optional live official-source probe → register both tools → Q8 paid run → save & compare → manual frozen checklist evaluation.

**Practical note:** EUR-Lex may respond with a JS/blocked page. Such a response is explicitly `unverified`; do not bypass restrictions or infer a date. This notebook has not performed any live EUR-Lex/API execution for you.
### v2 corrections
- Replaces EUR-Lex HTML HTTP 202/empty response with official Cellar SPARQL requests.
- The old `cdm:work_id_document` consolidation query was unsuitable; uses `cdm:resource_legal_id_celex` and two discovery paths.
- `0 rows` means **unverified**, never evidence that no later consolidation exists.
- Preserves NB03 retriever, benchmark, 50/50 gate, model, and NB04b context unchanged.
- Runtime network not validated in this authoring environment; Q8 official run only after real Cellar metadata is observed.


## 0. Dependencies and execution rules

Install packages only if missing. Avoid blanket upgrades in Anaconda. Restart the kernel after any install.

`OPENAI_API_KEY` is **not necessary** for the parity audit or EUR-Lex parser/local checks; only the final agent run asks for it.

In [ ]:
# Only when packages are missing:
# %pip install openai-agents sentence-transformers rank-bm25 numpy pandas requests beautifulsoup4

In [1]:
import os, re, json, hashlib, platform
from datetime import datetime, timezone, date
from pathlib import Path
from getpass import getpass
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from agents import Agent, Runner, ModelSettings, function_tool

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Run from Regulatory_agent repository root (data/ missing)."
MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"
print("Python:", platform.python_version(), "| Model:", MODEL_NAME)
print("API key available:", bool(os.getenv("OPENAI_API_KEY")))
print("API key is not required until the Q8 official run.")

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Python: 3.13.5 | Model: gpt-6-luna
API key available: False
API key is not required until the Q8 official run.


## 1. Load and fingerprint the **frozen** Build02 / NB04b assets

All compared outputs are loaded rather than regenerated. The Q8 checklist is **evaluation-only**, never given to the agent. Separate run artifacts use `build03_nb06_` prefixes.

In [2]:
FILES = {
    "chunks": DATA_DIR / "chunks_legal_aware.json",
    "benchmark": DATA_DIR / "benchmark_questions_v1.json",
    "retrieval_config": DATA_DIR / "retrieval_config_nb03_final_hybrid_static_rag.json",
    "nb03_results": DATA_DIR / "retrieval_results_nb03_final_hybrid_static_rag.csv",
    "nb04b_config": DATA_DIR / "nb04b_context_repaired_experiment_config.json",
    "nb04b_answers": DATA_DIR / "nb04b_context_repaired_openai_luna_answers_comparison.csv",
}
for key, path in FILES.items():
    assert path.is_file(), f"Missing {key}: {path}"

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"], encoding="utf-8-sig")
questions = benchmark["questions"]
Q8 = questions["Q8"]
benchmark_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert len(legal_chunks) == 1383, "Library changed: investigate before running controlled test"
assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["top_k"] == 5
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert set(comparison_df.qid) == set(questions)

FROZEN_Q8_ANSWER = comparison_df.loc[comparison_df.qid=="Q8", "rag_answer"].iloc[0]
print("Benchmark:", benchmark["benchmark_id"])
print("Chunks:", len(legal_chunks), "| Q8:", Q8)
print("Frozen NB04b Q8 answer loaded, characters:", len(FROZEN_Q8_ANSWER))

Benchmark: regulatory_agent_v1_notion_2026-10-08
Chunks: 1383 | Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?
Frozen NB04b Q8 answer loaded, characters: 2752


## 2. **Exact NB03 retriever implementation** (carry forward NB05 v2)

This implementation mirrors original NB03:
- searchable text = title/file/location/article/heading/text
- **BM25 tokenizer** = `[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?`
- normalized embeddings + dot-product, BM25Okapi
- Dense 50 + BM25 50, RRF k=60, source-group cap 2
- rank-ordered Top-5 for frozen test queries.

**No new BM25 or retrieval enhancement is introduced in NB06.**

In [3]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [safe(chunk.get(k)) for k in (
        "regulation_title", "file", "location", "article", "heading", "text"
    )]
    return " | ".join(p for p in parts if p).strip()

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

legal_texts = [searchable_text(ch) for ch in legal_chunks]
assert all(legal_texts)
EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent retriever components ready:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent retriever components ready: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    return (file_name, article if article else location)

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not (1 <= int(top_k) <= 10):
        raise ValueError("top_k must be in 1..10")
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]
    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)
    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append((idx, chunk, fused_scores[idx]))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break
    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank, "chunk_index": int(idx),
            "chunk_id": chunk.get("chunk_id"), "rrf_score": float(rrf_score),
            "file": chunk.get("file"), "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"), "location": chunk.get("location"),
            "heading": chunk.get("heading"), "text": chunk.get("text"),
        })
    return results

## 3. STOP-GATE · rank-ordered NB03 ↔ NB06 parity: **50/50**

Cannot continue to controlled Q8 if any chunk ID/order differs. Save a separate audit file; frozen NB03 outputs are read-only.

In [5]:
PARITY_PATH = DATA_DIR / "build03_nb06_retrieval_parity_audit.csv"
audit_rows, parity_by_qid = [], {}
for qid, question in questions.items():
    baseline = nb03_df[nb03_df.qid == qid].sort_values("rank")
    expected = baseline.chunk_id.astype(str).tolist()
    actual = [r["chunk_id"] for r in hybrid_search(question, top_k=5)]
    assert len(expected) == len(actual) == 5
    parity_by_qid[qid] = expected == actual
    for rank, (a,b) in enumerate(zip(expected,actual), start=1):
        audit_rows.append({"qid":qid,"rank":rank,"nb03_chunk_id":a,"nb06_chunk_id":b,"match":a==b})
    print(f"{qid}: {'PASS' if parity_by_qid[qid] else 'MISMATCH'}")
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(PARITY_PATH, index=False, encoding="utf-8-sig")
matched = int(audit_df.match.sum())
PARITY_OK = matched == 50 and all(parity_by_qid.values())
print(f"\nPARITY: {matched}/50 ordered chunks | Saved: {PARITY_PATH}")
assert PARITY_OK, "STOP: original NB03 retrieval parity failed. No official Q8 run allowed."

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks | Saved: data\build03_nb06_retrieval_parity_audit.csv


## 4. Frozen law registry

No changes to the five input laws, snapshot dates or original corpus. External lookups are restricted to official EU Publications Office Cellar identifiers.


In [6]:
LAW_REGISTRY = {
    "2023/915": {"celex":"32023R0915", "eli":"reg/2023/915", "snapshot_date":"2026-08-19", "snapshot_file":"EU_2023_915_consolidated_2026-08-19.html", "title":"Contaminants maximum levels"},
    "2019/1793": {"celex":"32019R1793", "eli":"reg_impl/2019/1793", "snapshot_date":"2026-06-30", "snapshot_file":"EU_2019_1793_consolidated_2026-06-30.html", "title":"Increased import controls / special conditions"},
    "2017/625": {"celex":"32017R0625", "eli":"reg/2017/625", "snapshot_date":"2025-01-05", "snapshot_file":"EU_2017_625_consolidated_2025-01-05.html", "title":"Official controls"},
    "178/2002": {"celex":"32002R0178", "eli":"reg/2002/178", "snapshot_date":"2026-01-01", "snapshot_file":"EU_2002_178_consolidated_2026-01-01.html", "title":"General Food Law"},
    "2019/1013": {"celex":"32019R1013", "eli":"reg_impl/2019/1013", "snapshot_date":None, "snapshot_file":"EU_2019__1013_original.html", "title":"Prior-notification arrangements"},
}

for key, item in LAW_REGISTRY.items():
    assert (DATA_DIR/item["snapshot_file"]).is_file(), f"Snapshot missing for {key}"
    item["official_basic_url"] = f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A{item["celex"]}'

assert set(LAW_REGISTRY) == {"2023/915","2019/1793","2017/625","178/2002","2019/1013"}
print("Current-law checker scope:", list(LAW_REGISTRY))

Current-law checker scope: ['2023/915', '2019/1793', '2017/625', '178/2002', '2019/1013']


## 5. Official Cellar SPARQL metadata client (v2)

EUR-Lex returned HTTP 202 with an empty HTML body in the previous run. This implementation queries official Cellar SPARQL for consolidated works by CELEX. The original v1 SPARQL draft mistakenly queried `work_id_document`; Cellar legal CELEX work identifiers use `resource_legal_id_celex`. It checks two retrieval paths and validates consolidated CELEX prefixes, dates, result shape, HTTP errors and redirects. **A latest date candidate is not proof of legal applicability or exhaustive coverage.**

**v3 correction:** A live diagnostic using `FILTER(CONTAINS(STR(?celex), "2019R1793"))` returned 19 records, including `02019R1793-20260630`. The prior `bif:contains` index query silently returned zero. The index query now uses the successful portable SPARQL pattern. A zero-row relation lookup is not itself an error, and an index match remains metadata evidence, not article-level legal verification.


In [7]:
# Official Cellar SPARQL: CELEX identifiers are stored under
# cdm:resource_legal_id_celex (NOT cdm:work_id_document).
# No EUR-Lex HTML scraping, no user-supplied URLs, no fabricated latest dates.
from urllib.parse import urlparse

CELLAR_SPARQL = "https://publications.europa.eu/webapi/rdf/sparql"
CELLAR_CELEX_BASE = "https://publications.europa.eu/resource/celex/"
CELLAR_TIMEOUT = (10, 45)
MAX_SPARQL_BYTES = 2_000_000
MAX_VERSIONS = 100


def _base_record(regulation):
    law = LAW_REGISTRY.get(regulation, {})
    return {
        "regulation": regulation, "basic_celex": law.get("celex"),
        "snapshot_date": law.get("snapshot_date"),
        "snapshot_file": law.get("snapshot_file"),
        "official_basic_url": law.get("official_basic_url"),
        "official_metadata_endpoint": CELLAR_SPARQL,
        "checked_utc": datetime.now(timezone.utc).isoformat(),
        "status": "unverified", "source": "EU Publications Office / Cellar SPARQL",
        "latest_date_candidate": None, "latest_celex_candidate": None,
        "observed_consolidations": [], "date_comparison": "not_verified",
        "article_level_changes_verified": False,
        "application_date_verified": False,
        "current_legal_status_verified": False,
        "complete_consolidation_list_verified": False,
    }


def _version_celex_pattern(basic_celex):
    # Sector 3 basic act -> sector 0 consolidated versions.
    return re.compile(r"^0" + re.escape(basic_celex[1:]) + r"-(\d{8})$")


def _sparql_versions_query(celex, mode="relation"):
    assert re.fullmatch(r"3\d{4}R\d{4}", celex), "Only allowlisted regulation CELEX IDs"
    prefix = "0" + celex[1:] + "-"
    header = "PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>\n"
    if mode == "relation":
        # Both documented relation predicates; incoming edges point at base work.
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?original cdm:resource_legal_id_celex "{celex}" .
  {{ ?consolidated cdm:act_consolidated_based_on_resource_legal ?original . }}
  UNION
  {{ ?consolidated cdm:act_consolidated_consolidates_resource_legal ?original . }}
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(STRSTARTS(STR(?consolidated_celex), "{prefix}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    if mode == "celex_index":
        # Validated against live Cellar: the plain SPARQL CONTAINS filter
        # returned 19 CELEX records (including 02019R1793-20260630).
        # The old Virtuoso bif:contains clause silently returned zero rows.
        # Search broadly, then use _parse_versions() to accept ONLY the
        # correct sector-0 consolidated CELEX IDs with valid dates.
        search_fragment = celex[1:]  # e.g. 2019R1793
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(CONTAINS(STR(?consolidated_celex), "{search_fragment}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    raise ValueError("Unknown SPARQL query mode")


def _run_cellar_sparql(query, session=None):
    client = session if session is not None else requests.Session()
    response = client.get(
        CELLAR_SPARQL, params={"query": query},
        headers={"Accept": "application/sparql-results+json",
                 "User-Agent": "ScientificAIBuilder-Research/0.2"},
        timeout=CELLAR_TIMEOUT, allow_redirects=True,
    )
    resolved = getattr(response, "url", CELLAR_SPARQL)
    parsed = urlparse(resolved)
    if parsed.scheme != "https" or parsed.hostname != "publications.europa.eu":
        raise ValueError("untrusted_redirect")
    if response.status_code != 200:
        raise ValueError(f"http_{response.status_code}")
    raw = response.content
    if not raw or len(raw) > MAX_SPARQL_BYTES:
        raise ValueError("empty_or_oversize_response")
    # Virtuoso can return HTTP 200 with plain-text query error.
    content_type = response.headers.get("Content-Type", "").lower()
    if "json" not in content_type:
        raise ValueError("non_json_or_sparql_error")
    data = response.json()
    if not isinstance(data.get("results", {}).get("bindings"), list):
        raise ValueError("invalid_sparql_result_structure")
    return data["results"]["bindings"]


def _parse_versions(bindings, basic_celex, as_of=None):
    as_of = as_of or datetime.now(timezone.utc).date()
    pattern = _version_celex_pattern(basic_celex)
    found = {}
    for row in bindings:
        value = row.get("consolidated_celex", {}).get("value", "")
        match = pattern.fullmatch(value)
        if not match:
            continue
        try:
            dt = datetime.strptime(match.group(1), "%Y%m%d").date()
        except ValueError:
            continue
        # Future-dated consolidations are recorded separately, not current.
        if dt > as_of:
            continue
        found[value] = {
            "celex": value,
            "date": dt.isoformat(),
            "official_celex_url": CELLAR_CELEX_BASE + value,
        }
    return sorted(found.values(), key=lambda v: (v["date"],v["celex"]), reverse=True)


def retrieve_cellar_current_law(regulation, session=None, as_of=None):
    """Official metadata check only; NEVER certify legal force or amended clauses.

    Relational lookup + independently scoped CELEX-index lookup.
    Missing results or failed confirmation are UNVERIFIED, not 'no update'.
    """
    rec = _base_record(regulation)
    if regulation not in LAW_REGISTRY:
        rec["error"] = "not_allowlisted"
        return rec
    basic = LAW_REGISTRY[regulation]["celex"]
    as_of = as_of or datetime.now(timezone.utc).date()
    if isinstance(as_of, str):
        as_of = date.fromisoformat(as_of)
    rec["as_of_date"] = as_of.isoformat()

    try:
        rel = _run_cellar_sparql(_sparql_versions_query(basic,"relation"),session)
        rec["relation_row_count"] = len(rel)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        rel = []
        rec["relation_error"] = f"{type(exc).__name__}: {exc}"

    # Important: index query is independent, not assumed equivalent to links.
    try:
        idx = _run_cellar_sparql(_sparql_versions_query(basic,"celex_index"),session)
        rec["celex_index_row_count"] = len(idx)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        idx = []
        rec["celex_index_error"] = f"{type(exc).__name__}: {exc}"

    observed = _parse_versions(rel+idx,basic,as_of=as_of)
    rec["observed_consolidations"] = observed
    rec["lookup_methods"] = ["relation", "celex_index"]
    if not observed:
        rec["error"] = "no_verified_consolidation_rows"
        return rec

    candidate = observed[0]
    rec["latest_date_candidate"] = candidate["date"]
    rec["latest_celex_candidate"] = candidate["celex"]
    rec["latest_candidate_official_url"] = candidate["official_celex_url"]

    # If either query failed, retrieval may be incomplete.
    if "relation_error" in rec or "celex_index_error" in rec:
        rec["status"] = "partial_metadata"
        rec["date_comparison"] = "latest_observed_only_incomplete_verification"
        return rec
    if len(rel) >= MAX_VERSIONS or len(idx) >= MAX_VERSIONS:
        rec["status"] = "partial_metadata"
        rec["error"] = "result_limit_reached"
        return rec

    rec["status"] = "metadata_found"
    rec["complete_consolidation_list_verified"] = False  # Graph freshness/coverage not provable here.
    snapshot = rec["snapshot_date"]
    if snapshot is None:
        rec["date_comparison"] = "original_snapshot_without_consolidation_date"
    elif candidate["date"] > snapshot:
        rec["date_comparison"] = "newer_consolidation_observed"
    elif candidate["date"] == snapshot:
        rec["date_comparison"] = "matches_latest_observed_consolidation"
    else:
        rec["date_comparison"] = "latest_observed_older_than_snapshot_review"
    return rec

# Alias preserves existing tool wiring below.
retrieve_eurlex_basic_page = retrieve_cellar_current_law


## 6. Offline Function Tests — no network or OpenAI API

Mocked SPARQL responses test normal matching, empty results, HTTP failures, cross-regulation data, future dates, and unknown IDs.


In [8]:
# Offline tests: these DO NOT establish real version metadata.
class _FakeResponse:
    def __init__(self, rows, status=200, url=CELLAR_SPARQL, bad_content_type=False):
        self.status_code = status
        self.url = url
        self.headers = {"Content-Type": "text/html" if bad_content_type else "application/sparql-results+json"}
        self._payload = {"head":{"vars":["consolidated_celex"]},"results":{"bindings":rows}}
        self.content = json.dumps(self._payload).encode("utf-8")
    def json(self): return self._payload

class _FakeSession:
    def __init__(self, responses):
        self.responses = list(responses)
    def get(self,*args,**kwargs):
        return self.responses.pop(0)

def _binding(celex):
    return {"consolidated_celex":{"type":"literal","value":celex}}

valid = _FakeSession([
    _FakeResponse([_binding("02019R1793-20260401")]),
    _FakeResponse([_binding("02019R1793-20260630"),_binding("02019R1793-20260401")]),
])
r = retrieve_cellar_current_law("2019/1793",session=valid,as_of="2026-10-19")
assert r["status"] == "metadata_found" and r["latest_date_candidate"] == "2026-06-30"
assert r["date_comparison"] == "matches_latest_observed_consolidation"
assert r["current_legal_status_verified"] is False

empty = _FakeSession([_FakeResponse([]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=empty,as_of="2026-10-19")
assert r["status"] == "unverified" and r["latest_date_candidate"] is None

blocked = _FakeSession([_FakeResponse([],status=202),_FakeResponse([],status=202)])
r = retrieve_cellar_current_law("2019/1793",session=blocked,as_of="2026-10-19")
assert r["status"] == "unverified"

wrong_law = _FakeSession([_FakeResponse([_binding("02023R0915-20260819")]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=wrong_law,as_of="2026-10-19")
assert r["status"] == "unverified"

future = _FakeSession([_FakeResponse([_binding("02019R1793-20270101")]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=future,as_of="2026-10-19")
assert r["status"] == "unverified"

assert retrieve_cellar_current_law("2099/9999")["error"] == "not_allowlisted"
print("PASS: Cellar SPARQL parsing, exact CELEX identity, dates and failure-closed behavior")

# Regression: use the successful Cellar SPARQL predicate, not Virtuoso full-text bif:contains.
index_query = _sparql_versions_query("32019R1793", "celex_index")
assert 'FILTER(CONTAINS(STR(?consolidated_celex), "2019R1793"))' in index_query
assert "bif:contains" not in index_query
print("PASS: CELEX index query matches successful live diagnostic")


PASS: Cellar SPARQL parsing, exact CELEX identity, dates and failure-closed behavior
PASS: CELEX index query matches successful live diagnostic


## 7. Live Cellar probe — real internet, no OpenAI API

This makes two targeted SPARQL requests to the EU Publications Office. A result of `unverified` is NOT evidence that no newer version exists. Preserve the JSON evidence and inspect errors.

**Expected for 2019/1793 (user live diagnostic):** index rows should include consolidated CELEX `02019R1793-20260630`; `latest_date_candidate` should equal `2026-06-30` if results remain unchanged.


In [9]:
# This DOES access the internet; no OpenAI API call.
RUN_LIVE_CELLAR_PROBE = True
LIVE_PROBE_PATH = DATA_DIR / "build03_nb06_v2_cellar_connectivity_probe.json"
if RUN_LIVE_CELLAR_PROBE:
    probe = retrieve_cellar_current_law("2019/1793")
    LIVE_PROBE_PATH.write_text(json.dumps(probe,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Cellar status:",probe["status"])
    print("Relation rows:",probe.get("relation_row_count"),"Index rows:",probe.get("celex_index_row_count"))
    print("Latest observed:",probe["latest_date_candidate"],"Snapshot:",probe["snapshot_date"])
    print("Comparison:",probe["date_comparison"])
    print("Relation error:",probe.get("relation_error"))
    print("Index error:",probe.get("celex_index_error"))
    print("Full evidence saved:",LIVE_PROBE_PATH)
else:
    print("Skipped live Cellar probe")


Cellar status: metadata_found
Relation rows: 0 Index rows: 19
Latest observed: 2026-06-30 Snapshot: 2026-06-30
Comparison: matches_latest_observed_consolidation
Relation error: None
Index error: None
Full evidence saved: data\build03_nb06_v2_cellar_connectivity_probe.json


In [10]:

import requests

url = (
    "https://eur-lex.europa.eu/legal-content/EN/TXT/"
    "?uri=CELEX:32019R1793"
)

try:
    response = requests.get(
        url,
        timeout=20,
        allow_redirects=True
    )

    print("HTTP status:", response.status_code)
    print("Final URL:", response.url)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("Retry-After:", response.headers.get("Retry-After"))
    print("Response length:", len(response.content))
    print("Response preview:")
    print(response.text[:500])

except requests.RequestException as e:
    print("Connection failed:", str(e))


HTTP status: 202
Final URL: https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32019R1793
Content-Type: text/html; charset=UTF-8
Retry-After: None
Response length: 0
Response preview:



## 8. Register the **two** Build03 Function Tools

`search_regulation()` is the exact same local NB03-equivalent hybrid search as NB05 v2.

`check_current_law(regulation)` is **only** an official Cellar metadata checker. It neither modifies the retriever nor pretends that an update in consolidation date proves substantive amendment or legal applicability. Every call is logged, including failure paths.

Both tools are independent. No hardcoded Q8 answers, no gold checklist injection.

In [11]:
RETRIEVAL_TOOL_LOG = []
CURRENT_LAW_TOOL_LOG = []

@function_tool
def search_regulation(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law EU regulation library using NB03-equivalent hybrid retrieval.

    Args:
        query: Focused legal search query.
        top_k: Number of ranked chunks to return (1–10, default 5).
    """
    top_k = max(1, min(int(top_k), 10))
    results = hybrid_search(query, top_k=top_k)
    RETRIEVAL_TOOL_LOG.append({"timestamp_utc":datetime.now(timezone.utc).isoformat(),"query":query,"top_k":top_k,"results":results})
    return json.dumps([{
        "rank":r["rank"],"chunk_id":r["chunk_id"],"file":r["file"],
        "article":r["article"],"location":r["location"],"heading":r["heading"],"text":r["text"]
    } for r in results], ensure_ascii=False)

@function_tool
def check_current_law(regulation: str) -> str:
    """Check official EU Publications Office Cellar consolidation metadata for one of five known EU regulations.

    A newer consolidation date is only a date observation, NOT proof that an article changed,
    a legal provision currently applies, or no other amendments exist. If blocked, report unverified.

    Args:
        regulation: Regulation identifier, e.g. '2019/1793', '2023/915', '2017/625', '178/2002', or '2019/1013'.
    """
    regulation = regulation.strip()
    rec = retrieve_cellar_current_law(regulation)
    CURRENT_LAW_TOOL_LOG.append(rec)
    return json.dumps(rec, ensure_ascii=False)

print("Tools ready: search_regulation, check_current_law")

Tools ready: search_regulation, check_current_law


## 9. Controlled Build03 Q8 agent

Same shared context, same answer headings, same model and reasoning effort as frozen NB04b. Differences: tool-use instructions and dynamic tool budget are the **Build03 intervention**. Q8 gold checklist and manually verified reference remain outside generation.

**No assertion of current-law validity from metadata-only evidence.** A law may be amended with distinct effective dates, transitional clauses, corrigenda, etc. If official website is unavailable, final answer must say which checks could not be completed.

In [12]:
assert PARITY_OK
SHARED_CONTEXT = nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT = nb04b_config["system_prompt"]
BUILD03_TOOL_RULES = """You have two tools: search_regulation (frozen snapshot legal text)
and check_current_law (official EU Cellar consolidation metadata for five known acts).
For material assertions, retrieve evidence. For Q8-style currentness analysis,
check official-source metadata when needed; report any unavailable/blocked checks as unverified.
State which topics require amendment, applicability-date, transitional-rule and annex-level checks.
A newer consolidated date does not prove changed provisions, legal effectiveness, or current applicability.
Do not infer absence of amendments from a failed or incomplete lookup.
Never assert an actual current limit, tariff code, check rate, or border action without specific verified evidence.
Use regulation numbers and official source links when provided by tools.
Be conservative; distinguish confirmed source observations from unverified legal questions.
""".strip()
AGENT_INSTRUCTIONS = "\n\n".join([SHARED_CONTEXT,SHARED_SYSTEM_PROMPT,BUILD03_TOOL_RULES])

regulatory_agent = Agent(
    name="Build03 Regulatory Research Agent NB06",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning={"effort":REASONING_EFFORT},
        max_tokens=int(nb04b_config["max_output_tokens"]),
    ),
    instructions=AGENT_INSTRUCTIONS,
    tools=[search_regulation, check_current_law],
)
print("Agent ready | model:", MODEL_NAME, "| reasoning:", REASONING_EFFORT)
print("Q8:", Q8)

Agent ready | model: gpt-6-luna | reasoning: medium
Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?


## 10. Official Q8 test — **paid API call + live Cellar requests**

**This cell incurs API costs and may make multiple Cellar network requests.** Jupyter runs SDK Agents with `await Runner.run`, not `run_sync`.

Guard against accidental overwriting: a completed run file will block rerun unless you deliberately archive/rename it. Network/tool logs are retained for independent audit.

Before running, ensure you can reach Cellar; if not, a run can still test *honest uncertainty handling*, but cannot demonstrate successful real-time verification.

In [13]:
Q8_RUN_PATH = DATA_DIR / "build03_nb06_q8_official.json"
assert PARITY_OK
# Prevent a paid official Q8 run if currentness tool cannot retrieve official metadata.
if not ("probe" in globals() and probe.get("status") == "metadata_found"):
    raise RuntimeError("Cellar probe did not verify a consolidation candidate. Do not run official Q8; first review the Cellar probe JSON.")
if Q8_RUN_PATH.exists():
    raise FileExistsError(f"A Q8 official run already exists: {Q8_RUN_PATH}. Archive it before deliberate rerun.")
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
assert os.getenv("OPENAI_API_KEY"), "API key required for Q8 run"

RETRIEVAL_TOOL_LOG.clear()
CURRENT_LAW_TOOL_LOG.clear()

q8_result = await Runner.run(regulatory_agent, Q8, max_turns=18)
q8_answer = str(q8_result.final_output)

# Save the complete tool records promptly, in the same execution cell.
q8_record = {
    "build":"Build03", "experiment":"q8_current_law_checker_official_nb06",
    "status":"run_completed_evaluation_pending",
    "benchmark_id":benchmark["benchmark_id"],"benchmark_sha256":benchmark_hash,
    "question_id":"Q8","question":Q8,"model":MODEL_NAME,
    "reasoning_effort":REASONING_EFFORT,
    "max_output_tokens":int(nb04b_config["max_output_tokens"]),
    "shared_benchmark_context":SHARED_CONTEXT,
    "shared_system_prompt":SHARED_SYSTEM_PROMPT,
    "build03_tool_rules":BUILD03_TOOL_RULES,
    "baseline_retrieval_config":retrieval_config,
    "parity":{"passed":PARITY_OK,"matched_ordered_chunks":matched,"total_ordered_chunks":50,"audit_file":str(PARITY_PATH)},
    "local_search_call_count":len(RETRIEVAL_TOOL_LOG),
    "current_law_call_count":len(CURRENT_LAW_TOOL_LOG),
    "local_search_tool_calls":RETRIEVAL_TOOL_LOG,
    "current_law_tool_calls":CURRENT_LAW_TOOL_LOG,
    "final_output":q8_answer,
    "run_at_utc":datetime.now(timezone.utc).isoformat(),
}
with Q8_RUN_PATH.open("x", encoding="utf-8") as f:
    json.dump(q8_record, f, ensure_ascii=False, indent=2)
print("Saved official Q8 run:", Q8_RUN_PATH)
print("\n===== BUILD03 Q8 FINAL ANSWER =====\n", q8_answer)
print("\nTool calls: local retrieval =", len(RETRIEVAL_TOOL_LOG), "; current law =", len(CURRENT_LAW_TOOL_LOG))
for c in CURRENT_LAW_TOOL_LOG:
    print(c["regulation"], "|", c["status"], "|",c.get("latest_date_candidate"),"|",c.get("date_comparison"),"|",c.get("error", ""))

Saved official Q8 run: data\build03_nb06_q8_official.json

===== BUILD03 Q8 FINAL ANSWER =====
 ## 1. Short answer

The conclusions most dependent on **current legislation** are the aflatoxin maximum level and product-category rules; whether this Türkiye-origin product is subject to special import controls and what those controls require; and the applicable border, documentation, sampling, and enforcement procedures. Each must be checked against the law applicable **on the relevant import or marketing date**.

## 2. Legal basis and relevant conditions

- **Contaminant limit and product category — Regulation (EU) 2023/915.** Check the current annex entry for dried figs and whether the product’s composition, intended use, or processing changes which entry applies. The frozen text distinguishes dried figs from some processed products and refers to a different rule for certain products that do not meet that category. Therefore the limit cannot be selected without both the current annex and

## 11. Frozen Build02 Q8 vs new Build03 Q8 (no baseline regeneration)

The previous NB04b Build02 result and official Notion grades stay fixed. Do **not** automatically claim `Yes` for correctness/grounding/completeness because a website returned a date. Review tool traces and the legal provisions separately.

In [14]:
print("===== FROZEN NB04b BUILD02 Q8 RAG =====\n")
print(FROZEN_Q8_ANSWER)
if Q8_RUN_PATH.exists():
    official = json.loads(Q8_RUN_PATH.read_text(encoding="utf-8"))
    print("\n===== BUILD03 NB06 Q8 =====\n")
    print(official["final_output"])
    print("\nOfficial-source outcomes:")
    for r in official["current_law_tool_calls"]:
        print(r["regulation"],r["status"],r.get("date_comparison"),r.get("official_basic_url"))
else:
    print("Q8 paid run not yet executed; frozen baseline still available.")

===== FROZEN NB04b BUILD02 Q8 RAG =====

## 1. Short answer

The parts of the assessment that depend on the law in force at the relevant time are the **applicable aflatoxin limits**, the **product’s use or processing classification**, and the **import-control requirements applying to dried figs from Türkiye**. Any conclusions about required certificates or attestations, official checks, or action following a result above a legal limit also need to be checked against the current rules.

The supplied evidence specifically shows that the import-control lists in Regulation (EU) 2019/1793 are reviewed regularly, at intervals not exceeding six months [E1]. It does **not** establish the current limit, whether this product-origin combination is currently listed, or what requirements apply to this shipment.

## 2. Legal basis and relevant conditions

- **Import-control listing and measures:** Check the current Annexes I, II and IIa of Regulation (EU) 2019/1793. The Commission must review those 

## 12. Frozen Q8 evaluation worksheet (human-reviewed, not model-generated)

**Q8 must-have checklist (Notion, 2026-10-08 frozen reference)**
1. Current 2023/915 limits, product scope and Art. 3 implications.
2. Current 2019/1793 Annex II, hazards, CN/TARIC codes and check frequency.
3. Entry documents/procedures (2019/1793 Arts 9–11 / Annex IV; 2017/625; 2019/1013).
4. Enforcement/current legal versions.
5. Assessment date, applicability dates, amendments and transitional provisions.
6. Static RAG is only a legal snapshot.

**Scoring:** `Correct` / `Grounded` / `Complete` each in `Yes`, `Partial`, `No`. Currentness verification *cannot* be scored fully grounded if all official lookups return `unverified`. A checked consolidated date is evidence of publication/version metadata, not article-level legal correctness.

This checklist is presented **after** generation, not provided to the agent.

In [15]:
EVALUATION_PATH = DATA_DIR / "build03_nb06_q8_evaluation_worksheet.csv"
if EVALUATION_PATH.exists():
    print("Existing evaluation worksheet preserved:", EVALUATION_PATH)
else:
    worksheet = pd.DataFrame([
        {"qid":"Q8", "build":"Build02_frozen", "correct":"Yes", "grounded":"Yes", "complete":"Partial", "note":"Notion frozen Q8 evaluation"},
        {"qid":"Q8", "build":"Build03_NB06", "correct":"", "grounded":"", "complete":"", "note":"HUMAN REVIEW: check original evidence + actual EUR-Lex tool outcomes"},
    ])
    worksheet.to_csv(EVALUATION_PATH,index=False,encoding="utf-8-sig")
    print("Created:", EVALUATION_PATH)
    display(worksheet)

Created: data\build03_nb06_q8_evaluation_worksheet.csv


,qid,build,correct,grounded,complete,note
0,Q8,Build02_frozen,Yes,Yes,Partial,Notion frozen Q8 evaluation
1,Q8,Build03_NB06,,,,HUMAN REVIEW: check original evidence + actual...


## 13. Final audit checklist and interpretation

- [ ] Same benchmark SHA-256, **1383** chunks and five frozen regulations.
- [ ] **50/50** ordered NB03 retrieval parity; STOP if not.
- [ ] Two SDK tools registered with `@function_tool`, no obsolete `@tool`.
- [ ] Offline parser tests pass and network availability separately recorded.
- [ ] `check_current_law` only returns **actual retrieved official metadata** (or explicit `unverified`).
- [ ] No invented current legal rates, limits or amendment claims.
- [ ] Q8 uses same NB04b benchmark context, model, reasoning effort and output format.
- [ ] Official Q8 result saved **separately**; old Q7 pilot/v2 and frozen NB04b never overwritten.
- [ ] Human review of Q8 checklist and verified source links before updating project conclusions.

**Research note:** With a second tool, Build03 differs from Build02 in both *retrieval policy* and *available official-source evidence*. Treat any improvement as tool-enabled evidence acquisition, not evidence of autonomous planning gains. Exact provision lookup remains an optional independent diagnostic for the Q7 local-library recall failure.

**What this v1 intentionally does not claim:** current legal validity, complete amendment history, verified effective dates, annex-row diff, exact jurisdictional enforcement applicability. Those require an additional structured amendment/current-law verification pipeline.